# AI 3D Scene Describer — dựng artifact trên Colab

Notebook này clone/lấy code, cài deps, chạy pipeline (procedural 3D + mô tả AI), xem
mô hình 3D ngay tại đây, rồi tải gói artifact về để bỏ vào `frontend/public/artifacts/`.

**Không cần GPU.** Procedural builder chỉ dùng CPU. (Phần generative GPU là stub, để mở rộng sau.)

Có 2 cách lấy code — chọn 1 ở Bước 1.

## Bước 1 — Lấy code
Đặt `REPO_URL` nếu đã đẩy lên GitHub. Để trống nếu muốn **upload file zip** (chứa thư mục `builder/` và `requirements.txt`).

In [ ]:
REPO_URL = ""   # ví dụ: "https://github.com/<user>/3D_Building.git"  |  để trống -> upload zip
BRANCH   = "main"

import os, glob, zipfile, io

def _find_repo_root(start="."):
    for path in glob.glob(os.path.join(start, "**", "requirements.txt"), recursive=True):
        root = os.path.dirname(path)
        if os.path.isdir(os.path.join(root, "builder")):
            return root
    return None

if REPO_URL:
    !rm -rf repo && git clone --depth 1 --branch $BRANCH $REPO_URL repo
    ROOT = _find_repo_root("repo") or "repo"
else:
    from google.colab import files
    print("Tải lên file .zip của repo (phải chứa builder/ và requirements.txt)...")
    up = files.upload()
    name = list(up)[0]
    with zipfile.ZipFile(io.BytesIO(up[name])) as z:
        z.extractall("repo")
    ROOT = _find_repo_root("repo")

assert ROOT, "Không tìm thấy thư mục chứa builder/ + requirements.txt"
os.chdir(ROOT)
print("Repo root:", os.getcwd())
print("Có builder/:", os.path.isdir("builder"))

## Bước 2 — Cài dependencies

In [ ]:
!pip install -q -r requirements.txt
import trimesh, pydantic
print("trimesh", trimesh.__version__, "| pydantic", pydantic.VERSION)

## Bước 3 (tuỳ chọn) — Dùng Claude thay cho Mock
Bỏ comment và dán API key để AI viết mô tả bằng Claude. Nếu bỏ qua, pipeline chạy hoàn toàn offline bằng MockLLM.

In [ ]:
# import os
# os.environ["LLM_PROVIDER"] = "claude"
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["CLAUDE_MODEL"] = "claude-sonnet-4-6"

## Bước 4 — Dựng từ prompt của bạn
Sửa các thông số bên dưới rồi chạy. Có thể để trống `floors/rooms/occupancy` và mô tả bằng ngôn ngữ tự nhiên (ví dụ: "8 tầng, mỗi tầng 5 phòng") — spec_agent sẽ tự suy ra.

In [ ]:
import json
from builder.pipeline import generate
from builder.schemas import GenerateRequest, SpaceType

req = GenerateRequest(
    project_name="Sunrise Office Tower",
    space_type=SpaceType.office,            # office | residential | retail | mixed | education
    description="Tòa văn phòng 5 tầng, mỗi tầng 6 phòng, khoảng 120 người",
    target_audience="Doanh nghiệp SME thuê văn phòng",
    floors=None, rooms_per_floor=None, occupancy=None,
)

bundle = generate(req, out_dir="artifacts")
print("id     :", bundle.id)
print("spec   :", bundle.spec.floors, "tầng x", bundle.spec.rooms_per_floor, "phòng")
print("model  :", bundle.model.glb, f"({bundle.model.tri_count} tris, {bundle.model.size_kb} KB)")
print("llm    :", bundle.meta.llm_provider, "| build", bundle.meta.build_ms, "ms")
print()
print(json.dumps(bundle.describer.model_dump(), ensure_ascii=False, indent=2))

## Bước 4b — Preset: tòa nhà kiểu GreenFlow (VinHack)

Dựng đúng bối cảnh dự án GreenFlow: tòa văn phòng nhỏ **OfficeSmall** tại TP.HCM gần
**Tân Sơn Nhất**, mỗi tầng có **lõi trung tâm (CORE_ZN)** bao quanh bởi các phòng chu vi,
dùng làm digital twin giám sát và tối ưu HVAC. (Gán vào `bundle` nên Bước 5 sẽ xem tòa này.)

In [ ]:
from builder.pipeline import generate
from builder.schemas import GenerateRequest, SpaceType

bundle = generate(GenerateRequest(
    project_name="GreenFlow OfficeSmall HCM",
    space_type=SpaceType.office,
    description=(
        "Tòa văn phòng nhỏ kiểu OfficeSmall tại TP.HCM, gần sân bay Tân Sơn Nhất. "
        "Mỗi tầng gồm một lõi trung tâm (CORE_ZN) bao quanh bởi các phòng làm việc chu vi. "
        "3 tầng, mỗi tầng 5 phòng, sức chứa khoảng 60 người. "
        "Tòa nhà dùng làm digital twin để giám sát và tối ưu vận hành HVAC."
    ),
    target_audience="Đội vận hành tòa nhà, chủ đầu tư và đội tối ưu năng lượng",
    floors=3, rooms_per_floor=5, occupancy=60,
), out_dir="artifacts")

print("id    :", bundle.id)
print("title :", bundle.describer.title)
print("tang 1:")
for r in bundle.structure.floors[0].rooms:
    print(f"  - {r.name} | {r.type} | {r.area} m2")

## Bước 5 — Xem mô hình 3D ngay trong Colab
Nhúng GLB bằng `<model-viewer>` (kéo để xoay).

In [ ]:
import base64
from IPython.display import HTML

glb_path = f"artifacts/{bundle.model.glb}"
b64 = base64.b64encode(open(glb_path, "rb").read()).decode()
HTML(f'''
<script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{b64}"
  camera-controls auto-rotate shadow-intensity="0.6" exposure="1.05"
  style="width:100%;height:460px;background:#f0eee6;border-radius:16px"></model-viewer>
''')

## Bước 6 (tuỳ chọn) — Dựng nhiều scene mẫu một lần

In [ ]:
samples = [
    dict(project_name="Sunrise Office Tower", space_type=SpaceType.office,
         description="Tòa văn phòng 5 tầng, mỗi tầng 6 phòng, khoảng 120 người",
         target_audience="Doanh nghiệp SME thuê văn phòng"),
    dict(project_name="Lake View Residences", space_type=SpaceType.residential,
         description="Chung cư 12 tầng, mỗi tầng 4 căn hộ, khoảng 200 cư dân",
         target_audience="Gia đình trẻ khu vực ven hồ"),
    dict(project_name="Centro Retail Hub", space_type=SpaceType.retail,
         description="Trung tâm bán lẻ 3 tầng, mỗi tầng 8 gian hàng, sức chứa 600 khách",
         target_audience="Thương hiệu bán lẻ và F&B"),
]
for s in samples:
    b = generate(GenerateRequest(**s), out_dir="artifacts")
    print("ok:", b.id, "|", b.model.tri_count, "tris")

## Bước 7 — Tải artifact về
Giải nén và copy toàn bộ vào `frontend/public/artifacts/` (gồm cả `index.json`), rồi deploy/commit để demo trên Vercel đọc.

In [ ]:
import shutil
shutil.make_archive("scene-artifacts", "zip", "artifacts")
from google.colab import files
files.download("scene-artifacts.zip")